# CDF PDF Extraction

This notebook shows a simple way to extract a table from a PDF using `pdfplumber`.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import pdfplumber

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.paths import RAW_PDF_DIR, RECONSTRUCTED_PDF_DIR, EXTRACTED_DIR

## 2. Choose A PDF

In [ ]:
pdf_files = sorted(RECONSTRUCTED_PDF_DIR.glob("*.pdf"))

if not pdf_files:
    pdf_files = sorted(RAW_PDF_DIR.glob("*.pdf"))

input_pdf = pdf_files[0] if pdf_files else None
input_pdf

## 3. Extract The First Table

In [ ]:
rows = []

if input_pdf is None:
    print("No PDF files found.")
else:
    with pdfplumber.open(input_pdf) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                rows = table
                break

    print("Rows extracted:", len(rows))

## 4. Save As CSV

In [ ]:
if rows:
    header = rows[0]
    data_rows = rows[1:]
    df = pd.DataFrame(data_rows, columns=header)

    output_file = EXTRACTED_DIR / f"{input_pdf.stem}.csv"
    number = 2
    while output_file.exists():
        output_file = EXTRACTED_DIR / f"{input_pdf.stem}_{number}.csv"
        number += 1

    df.to_csv(output_file, index=False, sep="|")
    print("Saved to:", output_file)
else:
    print("No table was extracted. Check the PDF manually.")